# From Waveform to Genre — Modeling & Training


**FMA dataset (citation)**  
Defferrard, M., Benzi, K., Vandergheynst, P., & Bresson, X. (2017). *FMA: A Dataset for Music Analysis*. 18th ISMIR. PDF: https://arxiv.org/pdf/1612.01840.pdf

**License / data note:** FMA metadata is CC BY 4.0; audio files follow per-artist Creative Commons licenses. Consult the original dataset for terms of use.

---

### Project adaptation note
This notebook adapts the FMA data pipeline for the current workflow and interactive environments (Google Colab / Drive). Key adaptations include Colab/Drive path variables, idempotent download & extraction steps, automatic manifest generation for reproducibility, and helper utilities for checksums and seeding. Any reuse of original code or logic from the mdeff/fma project is indicated in the header and documented in the repository.


***
This notebook prepares and trains an audio-based genre classifier end-to-end. It configures the Colab environment and project paths, defines audio preprocessing and dataset utilities, builds a PyTorch dataset/loader that converts raw audio into normalized Mel-spectrogram inputs, and implements the model training loop with checkpointing and training-history logging. After training, it runs inference on the test split, saves per-track predictions and evaluation metrics (confusion matrix, classification report), and produces diagnostics and visualizations — including learning curves, latent-space projections (t-SNE/UMAP) with silhouette scores, class distribution analysis, and example spectrograms for misclassified tracks. Finally, the notebook records provenance metadata and synchronizes the processed data and outputs to Google Drive.

### Short comparison vs. original mdeff/fma
1. Focused on end-to-end model training in Colab (audio waveform → mel-spectrogram → CNN) rather than the original repo's feature-centric tooling.

2. Adds Colab/Drive integration, automated metadata download + SHA-1 verification, balanced per-genre sampling, and stratified train/val/test splits.

3. Implements data provenance saving and Drive sync (overwrites previous copy) for reproducibility.

4. Includes inference, evaluation, visualization (t-SNE/UMAP, confusion matrices, learning curves) and error-analysis pipelines not present in the original.

### Mount Drive and Configure Project Paths
This mounts Google Drive in Colab, sets up project and data directories, ensures the local data folder exists, changes the working directory to the project folder, and prints the local data path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
WORKDIR = "/content/waveform_genre_project"
LOCAL_DATA = os.path.join(WORKDIR, "data_storage")
DRIVE_ROOT = "/content/drive/MyDrive/waveform_analysis_outputs"
os.makedirs(LOCAL_DATA, exist_ok=True)
os.chdir(WORKDIR)
print("WORKDIR set, LOCAL_DATA:", LOCAL_DATA)

Sets up required libraries and a compact modeling configuration (audio paths and preprocessing params, random seed, training hyperparameters, and output file locations) for training an audio genre classifier.

In [ ]:
# standard libs for filesystem, archives, downloads, and utility operations
import os, zipfile, urllib.request, shutil, time, datetime
# data libraries for arrays and tabular data
import numpy as np, pandas as pd
# PyTorch for model definition and training
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
# audio processing library used to load and convert audio to spectrograms
import librosa

# Modeling config (kept compact)
# URL to the small FMA audio archive (download source for audio files)
AUDIO_URL = "https://os.unil.cloud.switch.ch/fma/fma_small.zip"
# local path where the downloaded audio ZIP will be stored
AUDIO_ZIP_LOCAL = os.path.join(LOCAL_DATA, "fma_small.zip")
AUDIO_DIR = os.path.join(LOCAL_DATA, "audio")
# audio preprocessing parameters: sampling rate and clip length (seconds)
SAMPLE_RATE = 22050
CLIP_SECONDS = 10
# spectrogram parameters: number of mel bands, FFT window size, and hop length
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
# reproducibility: fixed random seed for sampling/splitting/initialization
SEED = 42
# training hyperparameters: batch size, learning rate, and number of epochs
BATCH_SIZE = 16
LR = 5e-4
EPOCHS = 1   # demo short training by default
# paths for saving model weights and training history (CSV)
MODEL_PATH = os.path.join(LOCAL_DATA, "genre_classifier_net.pt")
HISTORY_CSV = os.path.join(LOCAL_DATA, "training_history.csv")